In [6]:
import pandas as pd
import numpy as np

df={
    'customers': pd.read_csv('../../Olist_Data/olist_customers_dataset.csv'),
    'orders': pd.read_csv('../../Olist_Data/olist_orders_dataset.csv'),
    'order_items': pd.read_csv('../../Olist_Data/olist_order_items_dataset.csv'),
    'payments': pd.read_csv('../../Olist_Data/olist_order_payments_dataset.csv'),
    'reviews': pd.read_csv('../../Olist_Data/olist_order_reviews_dataset.csv'),
    'products': pd.read_csv('../../Olist_Data/olist_products_dataset.csv'),
    'sellers': pd.read_csv('../../Olist_Data/olist_sellers_dataset.csv'),
    'geolocation': pd.read_csv('../../Olist_Data/olist_geolocation_dataset.csv')
}

print("Checking Missing values in the datasets")

for name,table in df.items():
    null_count = table.isnull().sum()
    null_percent = table.isnull().mean() * 100
    
    missing_col=null_count[null_count > 0]
    
    if len(missing_col) > 0:
        print(f"Table : {name}")
        print(pd.DataFrame({'missing_count': null_count[missing_col.index], 'missing_pct': null_percent[missing_col.index]}))
        


Checking Missing values in the datasets
Table : orders
                               missing_count  missing_pct
order_approved_at                        160     0.160899
order_delivered_carrier_date            1783     1.793023
order_delivered_customer_date           2965     2.981668
Table : reviews
                        missing_count  missing_pct
review_comment_title            87656    88.341530
review_comment_message          58247    58.702532
Table : products
                            missing_count  missing_pct
product_category_name                 610     1.851234
product_name_lenght                   610     1.851234
product_description_lenght            610     1.851234
product_photos_qty                    610     1.851234
product_weight_g                        2     0.006070
product_length_cm                       2     0.006070
product_height_cm                       2     0.006070
product_width_cm                        2     0.006070


In [7]:
# ---------- ORDERS TABLE ----------
# order_approved_at missing (0.16%) - probably approved right after purchase
# type: MAR - depends on order status
# fix: just use purchase time instead
df['orders']['order_approved_at'] = df['orders']['order_approved_at'].fillna(df['orders']['order_purchase_timestamp'])
# order_delivered_carrier_date missing (1.79%) - order was never shipped
# type: MAR - tied to order status
# fix: don't guess a date, just flag it
df['orders']['carrier_date_missing'] = df['orders']['order_delivered_carrier_date'].isnull().astype(int)
# order_delivered_customer_date missing (2.98%) - order was never delivered
# type: MAR - same reason as above
# fix: keep it empty, just flag it
df['orders']['customer_date_missing'] = df['orders']['order_delivered_customer_date'].isnull().astype(int)

# ---------- REVIEWS TABLE ----------
# review_comment_title missing (88.3%) - customer just didn't write a title
# type: MAR - depends on whether they wrote anything at all
# fix: fill with "no_title" and flag it
df['reviews']['title_missing'] = df['reviews']['review_comment_title'].isnull().astype(int)
df['reviews']['review_comment_title'] = df['reviews']['review_comment_title'].fillna('no_title')


# review_comment_message missing (58.7%) - same idea, no comment left
# type: MAR
# fix: fill with "no_comment" and flag it
df['reviews']['comment_missing'] = df['reviews']['review_comment_message'].isnull().astype(int)
df['reviews']['review_comment_message'] = df['reviews']['review_comment_message'].fillna('no_comment')

# ---------- PRODUCTS TABLE ----------
# product_category_name missing (1.85%) - some products just never got categorized
# type: MNAR - could be tied to the product itself being obscure/incomplete
# fix: fill with "unknown"
df['products']['product_category_name'] = df['products']['product_category_name'].fillna('unknown')


# product_name_lenght, product_description_lenght, product_photos_qty missing (same 1.85%)
# these are missing on the same rows as category, so probably incomplete listings
# type: MAR
# fix: fill with median
df['products']['product_name_lenght'] = df['products']['product_name_lenght'].fillna(df['products']['product_name_lenght'].median())
df['products']['product_description_lenght'] = df['products']['product_description_lenght'].fillna(df['products']['product_description_lenght'].median())
df['products']['product_photos_qty'] = df['products']['product_photos_qty'].fillna(df['products']['product_photos_qty'].median())

# weight/length/height/width missing (only 2 rows each, tiny)
# type: MCAR - too few rows to see a pattern, probably just random data entry gap
# fix: fill with median, barely affects anything
df['products']['product_weight_g'] = df['products']['product_weight_g'].fillna(df['products']['product_weight_g'].median())
df['products']['product_length_cm'] = df['products']['product_length_cm'].fillna(df['products']['product_length_cm'].median())
df['products']['product_height_cm'] = df['products']['product_height_cm'].fillna(df['products']['product_height_cm'].median())
df['products']['product_width_cm'] = df['products']['product_width_cm'].fillna(df['products']['product_width_cm'].median())


In [8]:
for name, table in df.items():
    counts = table.isnull().sum()
    missing = counts[counts > 0]
    if len(missing) > 0:
        print(f"\n{name} still has missing values:")
        print(missing)
    else:
        print(f"{name}: no missing values")

customers: no missing values

orders still has missing values:
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
order_items: no missing values
payments: no missing values
reviews: no missing values
products: no missing values
sellers: no missing values
geolocation: no missing values
